In [ ]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="YOUR_ACTUAL_KEY")
project = rf.workspace("nurs-workspace-uplim").project("benchmark-qqlzu")
version = project.version(4)
dataset = version.download("yolov8")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 285.0/285.0 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 47.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 87.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.4/58.4 kB 4.4 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 5.0.0.93
    Uninstalling opencv-python-headless-5.0.0.93:
      Successfully uninstalled opencv-python-headless-5.0.0.93
  Attempting uninstall: typer
    Found existing installation: typer 0.26.8
    Uninstalling typer-0.26.8:
      Successfully uninstalled typer-0.26.8
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to benchmark-4 in yolov8:: 100%|██████████| 49/49 [00:00<00:00, 5082.75it/s]


In [ ]:
import cv2
import numpy as np
import glob
import os

# Path to your downloaded benchmark validation or train images & labels
dataset_path = dataset.location
images_dir = os.path.join(dataset_path, 'train', 'images')
labels_dir = os.path.join(dataset_path, 'train', 'labels')

reference_pixel_areas = []

# Loop through all benchmark images
for img_file in glob.glob(os.path.join(images_dir, '*.jpg')):
    # Load image to get dimensions (height, width)
    img = cv2.imread(img_file)
    h, w, _ = img.shape

    # Corresponding label file
    base_name = os.path.splitext(os.path.basename(img_file))[0]
    label_file = os.path.join(labels_dir, f"{base_name}.txt")

    if os.path.exists(label_file):
        with open(label_file, 'r') as f:
            lines = f.readlines()

        for line in lines:
            parts = line.strip().split()
            class_id = int(parts[0])

            # Assuming class 0 (or whichever index your 1cm square is) is the benchmark square
            coords = [float(p) for p in parts[1:]]
            pts = np.array(coords).reshape(-1, 2)

            # Convert normalized coordinates back to pixel values
            pts[:, 0] *= w
            pts[:, 1] *= h
            pts = pts.astype(np.int32)

            # Create a blank mask and fill the polygon to count exact pixels
            mask = np.zeros((h, w), dtype=np.uint8)
            cv2.fillPoly(mask, [pts], 1)

            # Sum pixels inside the mask
            pixel_count = np.sum(mask)
            reference_pixel_areas.append(pixel_count)

# Calculate the final height-corrected conversion factor
if len(reference_pixel_areas) > 0:
    avg_pixels_per_cm2 = np.mean(reference_pixel_areas)

    # Camera geometry heights (in cm)
    H_camera = 14.0  # total distance from lens to the floor
    h_wound = 3.0    # height of the rat's back / wound from the floor

    d_bench = H_camera               # distance to benchmark on floor
    d_wound = H_camera - h_wound     # distance to elevated wound

    # Base floor conversion factor (cm² per pixel at floor level)
    base_conversion_factor = 1.0 / avg_pixels_per_cm2

    # Height correction ratio (optical scaling based on distance squared)
    height_correction_ratio = (d_wound / d_bench) ** 2

    # Final true conversion factor for the wound
    true_wound_conversion_factor = base_conversion_factor * height_correction_ratio

    print(f"--- CALCULATION SUCCESSFUL ---")
    print(f"Total benchmark images processed: {len(reference_pixel_areas)}")
    print(f"Average Floor Pixel Area for 1 cm²: {avg_pixels_per_cm2:.2f} pixels")
    print(f"Floor Base Conversion Factor: {base_conversion_factor:.8f} cm²/pixel")
    print(f"Height Correction Ratio (d_wound/d_bench)²: {height_correction_ratio:.4f}")
    print(f"True Wound Conversion Factor: 1 pixel = {true_wound_conversion_factor:.8f} cm²")
else:
    print("No label files found. Check your dataset path.")

--- CALCULATION SUCCESSFUL ---
Total benchmark images processed: 18
Average Floor Pixel Area for 1 cm²: 12135.44 pixels
Floor Base Conversion Factor: 0.00008240 cm²/pixel
Height Correction Ratio (d_wound/d_bench)²: 0.6173
True Wound Conversion Factor: 1 pixel = 0.00005087 cm²


In [ ]:
h, w, _ = img.shape
print(f"Roboflow exported image resolution: {w} x {h}")  # <-- add this line

Roboflow exported image resolution: 1280 x 720
